# Inferring with Large Language Models: Sentiment, Entities & Zero-Shot Classification

In traditional machine learning workflows, "inferring" tasks—such as sentiment analysis, entity extraction, customer emotion detection, and topic tagging—required:
1. Curating and labeling bespoke datasets for each distinct task.
2. Training and fine-tuning separate supervised models (e.g., standard BERT or classifiers).
3. Deploying and maintaining multiple distinct inference endpoints in the cloud.

With Large Language Models, a single unified foundation model and API can execute all of these analytical tasks simply by altering the prompt instructions.

### Core Concepts Covered:
- **Sentiment & Emotion Classification:** Extracting binary sentiment and detecting granular emotional signals (e.g., customer anger for escalation).
- **Named Entity & Attribute Extraction:** Pulling structured product and brand information into JSON.
- **Multi-Task Extraction:** Consolidating classification, boolean flags, and named entities into a single LLM API call.
- **Topic Inferencing & Zero-Shot Classification:** Extracting emergent themes and indexing articles against predefined topic lists to drive automated business alerts.

## 1. Environment Setup & Configuration

Configure credentials and initialize the client.
- Compatible with standard OpenAI API keys (`OPENAI_API_KEY`).
- Compatible with Google Gemini via its OpenAI-compatible endpoint (`GEMINI_API_KEY` pointing to `https://generativelanguage.googleapis.com/v1beta/openai/`).

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# 1. Load API key from .env file
_ = load_dotenv(find_dotenv())

# 2. Initialize the OpenAI client pointing to the working Gemini endpoint
client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

### Helper Function: `get_completion`
Encapsulates chat completion requests. `temperature=0` enforces deterministic, reproducible classifications.

In [2]:
# # 3. helper function to make it easier to get completions from the model
# def get_completion(prompt, model="gemini-3.8-flash"):
#     messages = [{"role": "user", "content": prompt}]
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0,  # 0 for deterministic output
#     )
#     return response.choices[0].message.content

# 3. Define prioritized models (from fastest/latest to reliable backups)
from openai import OpenAI, APIError, RateLimitError, InternalServerError, NotFoundError
DEFAULT_MODELS = [
    "gemini-pro-latest",
    "gemini-flash-latest",
    "gemini-flash-lite-latest",
    "gemini-2.0-flash-lite",
    "gemini-3.8-flash",
    "gemini-2.5-flash",
    "gemini-2.0-flash",
    "gemini-1.5-flash",
    "gemini-1.5-pro",
]

def get_completion(prompt, models=DEFAULT_MODELS, max_retries_per_model=2):
    """
    Attempts to get a completion by cascading through a prioritized list of models.
    Handles 429 (Quota/Rate Limits), 503 (High Demand/Unavailable), and 404 (Not Found).
    """
    # If a single model string is passed by mistake, wrap it in a list
    if isinstance(models, str):
        models = [models] + [m for m in DEFAULT_MODELS if m != models]

    messages = [{"role": "user", "content": prompt}]
    last_exception = None

    for model_name in models:
        delay = 1.0  # Initial delay for temporary errors

        for attempt in range(max_retries_per_model):
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=messages,
                    temperature=0,  # Deterministic output
                )
                return response.choices[0].message.content

            except RateLimitError as e:
                # 429 Quota Exceeded (Per-model limit reached) -> Switch immediately
                print(f"[429 Quota Exceeded] '{model_name}' limit reached. Switching to next model...")
                last_exception = e
                break  # Break inner loop to try next model immediately

            except NotFoundError as e:
                # 404 Model Not Found/Retired -> Switch immediately
                print(f"[404 Not Found] '{model_name}' is unavailable on this API version. Switching...")
                last_exception = e
                break

            except (InternalServerError, APIError) as e:
                # 503 / 500 Temporary Server Overload -> Retry once or switch
                last_exception = e
                status_code = getattr(e, "status_code", None)

                if attempt < max_retries_per_model - 1:
                    print(f"[{status_code or 'Error'}] '{model_name}' busy. Retrying in {delay}s...")
                    time.sleep(delay)
                    delay *= 2  # Exponential backoff
                else:
                    print(f"'{model_name}' failed after {max_retries_per_model} attempts. Switching to next model...")

    # If all models in the list fail:
    raise RuntimeError(f"All configured models failed. Last error: {last_exception}")

###### messages = [{"role": "user", "content": prompt}]: Formats your input prompt into the standard Chat Completion format, indicating that a human user is asking the question.

###### client.chat.completions.create(...): Sends the request to Gemini's server.

###### temperature=0: Sets the degree of randomness. A temperature of 0 ensures the model gives consistent, deterministic, and factual outputs rather than creative variations.

###### return response.choices[0].message.content: Extracts the generated text from the model's response and returns it as a string.

## 2. Source Data: Customer Review Sample

We use a customer review for a bedroom lamp that balances shipping damage, customer service resolution, product quality, and positive brand sentiment.

In [ ]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had 
additional storage and not too high of a price point. 
Got it fast. The string to our lamp broke during the 
transit and the company happily sent over a new one. 
Came within a few days as well. It was easy to put 
together. I had a missing part, so I contacted their 
support and they very quickly got me the missing piece! 
Lumina seems to me to be a great company that cares 
about their customers and products!!
"""

## 3. Sentiment Classification & Post-Processing Optimization

### Step 1: Open-Ended Classification
An unconstrained classification prompt returns a conversational sentence (e.g., *"The sentiment of the product review is positive."*).

In [ ]:
prompt_sentiment_open = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""

response_open = get_completion(prompt_sentiment_open)
print("--- Open-Ended Output ---")
print(response_open)

### Step 2: Constrained Single-Word Token Classification
For programmatic ingestion and downstream databases, constrain the output to a strict token (`positive` or `negative`).

In [ ]:
prompt_sentiment_constrained = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" 
or "negative".

Review text: '''{lamp_review}'''
"""

response_constrained = get_completion(prompt_sentiment_constrained)
print("--- Constrained Output ---")
print(response_constrained)

## 4. Emotional Tone Detection & Support Escalation

In customer operations, detecting nuanced emotions—specifically **anger** or severe dissatisfaction—allows automated triage systems to escalate tickets to high-touch support agents before churn occurs.

### 4.1 Granular Emotion Extraction

In [ ]:
prompt_emotions = f"""
Identify a list of emotions that the writer of the 
following review is expressing. Include no more than 
five items in the list. Format your answer as a list of 
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""

response_emotions = get_completion(prompt_emotions)
print("--- Extracted Emotions ---")
print(response_emotions)

### 4.2 Targeted Anger Detection (Binary Escalation Trigger)

In [ ]:
prompt_anger = f"""
Is the writer of the following review expressing anger?
The review is delimited with triple backticks. 
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""

response_anger = get_completion(prompt_anger)
print("--- Expressing Anger? ---")
print(response_anger)

## 5. Named Entity & Attribute Extraction into Structured JSON

Information extraction allows businesses to catalog items and manufacturers across thousands of unstructured customer reviews to track quality trends per vendor.

In [ ]:
prompt_entities = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. 
Format your response as a JSON object with 
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" 
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""

response_entities = get_completion(prompt_entities)
print("--- Extracted Entity JSON ---")
print(response_entities)

## 6. Multi-Task Inferencing in a Single Prompt

### Latency & Cost Optimization
Rather than issuing 4 separate API calls (`sentiment`, `anger`, `item`, `brand`), combine all analytical questions into a single consolidated prompt. 

Specify explicit data types (e.g., boolean for anger, lowercase strings for entities) to ensure direct deserialization into Python objects.

In [ ]:
prompt_multitask = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. 
Format your response as a JSON object with 
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" 
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""

response_multitask = get_completion(prompt_multitask)
print("--- Multi-Task Consolidated JSON ---")
print(response_multitask)

# Clean up markdown wrapping if present and parse into Python dictionary
clean_json = response_multitask.strip().strip("`").replace("json", "").strip()
parsed_data = json.loads(clean_json)
print("\nParsed Python Object:", parsed_data)
print("Anger Boolean Type:", type(parsed_data["Anger"]), "Value:", parsed_data["Anger"])

## 7. Topic Inferencing & Zero-Shot Classification

### News Article Dataset

In [3]:
story = """
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
"""

### 7.1 Open Topic Extraction
Extract the top 5 emerging themes from the text without supplying predefined categories.

In [ ]:
prompt_topics = f"""
Determine five topics that are being discussed in the 
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas.

Text sample: '''{story}'''
"""

response_topics = get_completion(prompt_topics)
topic_list_extracted = [t.strip() for t in response_topics.split(",")]
print("--- Top 5 Extracted Topics ---")
print(topic_list_extracted)

### 7.2 Zero-Shot Topic Indexing & News Alert Pipeline

In **Zero-Shot Learning**, the model classifies text into predefined categories without requiring prior training examples.

Below, we test whether our predefined target categories are present and trigger an automated downstream notification when a target theme (e.g., `nasa`) appears.

In [5]:
import json
target_topics = [
    "nasa", "local government", "engineering", 
    "employee satisfaction", "federal government"
]

# Production-grade zero-shot classifier using JSON schema
prompt_zero_shot = f"""
Determine whether each item in the following list of 
topics is a topic discussed in the text below delimited by triple backticks.

List of topics: {', '.join(target_topics)}

Provide your answer as a JSON object where each topic from the list is a key, 
and the value is 1 if the topic is covered, or 0 if it is not.

Text sample: '''{story}'''
"""

response_zero_shot = get_completion(prompt_zero_shot)
print("--- Raw Zero-Shot Classification ---")
print(response_zero_shot)

# Parse JSON and trigger automated business logic
clean_zero_shot_json = response_zero_shot.strip().strip("`").replace("json", "").strip()
topic_flags = json.loads(clean_zero_shot_json)

print("\nTopic Flags:", topic_flags)
if topic_flags.get("nasa") == 1:
    print("\n🔔 ALERT: New NASA story detected!")

[429 Quota Exceeded] 'gemini-pro-latest' limit reached. Switching to next model...
--- Raw Zero-Shot Classification ---
```json
{
  "nasa": 1,
  "local government": 0,
  "engineering": 0,
  "employee satisfaction": 1,
  "federal government": 1
}
```

Topic Flags: {'nasa': 1, 'local government': 0, 'engineering': 0, 'employee satisfaction': 1, 'federal government': 1}

🔔 ALERT: New NASA story detected!


## 8. Summary & Key Engineering Takeaways

| Task | Traditional ML Workflow | LLM Prompt Engineering Approach |
| :--- | :--- | :--- |
| **Sentiment & Emotions** | Curate labeled text, train BERT model, maintain microservice. | Single prompt with token constraints (`"positive"` or `"negative"`). |
| **Entity Extraction** | Build custom NER pipelines, annotate training data. | Specify JSON schema with keys (`Item`, `Brand`) and default fallback (`"unknown"`). |
| **Multi-Task Extraction** | Orchestrate 3-4 separate classification services. | Consolidate into 1 prompt call returning typed JSON (`boolean`, `string`). |
| **Zero-Shot Topic Indexing** | Train multiclass/multilabel models across domain taxonomies. | Provide target topic list in prompt and request binary flags (`0` or `1`) per topic. |